# Auditing a regression model with `c4fairness` (student grades)

Same idea, continuous target. A model predicts a student's final grade; the error is the signed
residual `y_true - y_pred`. `c4fairness` clusters the students and reports the **median error**
per cluster, so you can see where predictions are biased or unreliable and which students those
clusters contain.

Data: `Data/student_performance.csv` (670 students, with the model's `y_pred`). Sensitive
attributes span all three kinds: **binary** `sex_F`, **multi-categorical** `Medu` (mother's
education, 0–4), **numeric** `age`.

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("../Data/student_performance.csv")
df[["G1", "G2", "studytime", "absences", "sex_F", "Medu", "age", "y_true", "y_pred"]].head()

## 1. The residual

For regression the error is signed: `residual = y_true - y_pred`. Positive means the model
**under**-predicted the grade. Overall it should be near zero; the question is whether it is
near zero *within every cluster*.

In [ ]:
residual = df["y_true"] - df["y_pred"]
print(f"overall mean residual = {residual.mean():+.3f}   (bias)")
print(f"overall mean |residual| = {residual.abs().mean():.3f}   (typical error size)")

## 2. Encode + cluster

`Medu` is an integer code (0–4); we mark it categorical so it's analysed as categories, not as
a number. `age` stays continuous.

In [ ]:
from c4fairness.preprocessing import encode_categoricals
from c4fairness.clustering import cluster

sensitive = ["sex_F", "Medu", "age"]
col_lists = {"regular": ["G1", "G2", "studytime", "absences"], "sensitive": sensitive,
             "proxy": [], "special": []}
orig_sensitive = list(sensitive)

dfe, cl, cat_names, mcd, ohe = encode_categoricals(
    df.copy(), col_lists, ["Medu"], "kmeans", distance="euclidean"
)
clustering_cols = cl["regular"] + cl["sensitive"]
res = cluster(dfe[clustering_cols], algorithm="kmeans", distance="euclidean",
              n_clusters=3, random_state=42)
print(f"k = {res.n_clusters}   silhouette = {res.silhouette:.3f}   sizes = {res.cluster_sizes}")

## 3. Per-cluster recap

`error_type="regression"` makes the recap report the signed median error (`error_mean`, the
bias direction) and its magnitude (`abs_error_mean`), with ANOVA / Mann-Whitney significance
instead of Fisher. Sensitive features are shown salient (`Medu_cat` = dominant maternal
education level).

In [ ]:
from c4fairness.cli import _build_sensitive_analysis_list, apply_salient_reconstruction
from c4fairness.experiments import make_recap

analysis = _build_sensitive_analysis_list(cl["sensitive"], mcd, orig_sensitive, option="salient")
dfe["residual"] = residual.values
res_df = dfe.copy()
res_df["clusters"] = res.labels
apply_salient_reconstruction(res_df, mcd, orig_sensitive)

recap = make_recap(res_df, clustering_cols, sensitive_cols=analysis,
                   error_col="residual", error_type="regression",
                   feature_matrix=res.feature_matrix, continuous_sensitive_cols=["age"])
recap.round(3)

## 4. Reading it

- **`error_mean`** — the cluster's median signed residual: sign = direction of bias
  (over/under-prediction), magnitude = how biased.
- **`abs_error_mean`** — typical error size in the cluster (reliability).
- **`error_gap` / `error_gap_sig`** — difference vs. the rest and its significance
  (Mann-Whitney, one-vs-all).
- **`Medu_cat` / `sex_F_value` / `age_value`** — who the cluster contains.

A cluster with a large `|error| mean` and a significant `error_gap_sig` is where the model is
least reliable; its sensitive columns show for whom.

In [ ]:
from c4fairness.result_viz import plot_cluster_recap_heatmap
from IPython.display import Image

plot_cluster_recap_heatmap(recap.copy(), "student_residual", ".", error_label="residual")
Image("student_residual.png")